# Top-4 Custom Attention Multiseed Summary

This notebook reads aggregate CSV files produced by `aggregate_multiseed_results.py`. It does not train or evaluate models.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'custom_attention_multiseed' / 'run_top4_multiseed.py').exists():
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root()
RESULTS = REPO_ROOT / 'custom_attention_multiseed' / 'results'
summary_path = RESULTS / 'top4_multiseed_summary.csv'
sweep_path = RESULTS / 'top4_threshold_sweep.csv'
best_threshold_path = RESULTS / 'top4_threshold_best_by_module.csv'
ranking_path = RESULTS / 'top4_final_ranking.md'

print('Repo root:', REPO_ROOT)
print('Results directory:', RESULTS)
print('Summary exists:', summary_path.exists())
print('Sweep exists:', sweep_path.exists())


In [ ]:
summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
best_thresholds = pd.read_csv(best_threshold_path) if best_threshold_path.exists() else pd.DataFrame()
sweep = pd.read_csv(sweep_path) if sweep_path.exists() else pd.DataFrame()

if summary.empty:
    print('No summary rows yet. Run train, sweep, then aggregate first.')
else:
    display_cols = [
        'module', 'n_seeds', 'missing_seeds',
        'healthy_aware_score_mean', 'healthy_aware_score_std',
        'labeled_test_mask_map50_mean', 'labeled_test_mask_map50_std',
        'labeled_test_mask_map50_95_mean',
        'full_test_mask_map50_mean', 'full_test_mask_map50_std',
        'healthy_test_healthy_mask_fp_rate_mean', 'healthy_test_healthy_mask_fp_rate_std',
        'labeled_test_disease_box_miss_rate_mean', 'labeled_test_disease_box_miss_rate_std',
        'stability_std_sum', 'deployment_rank_sum', 'complete_3seed',
    ]
    display(summary[[c for c in display_cols if c in summary.columns]].sort_values('healthy_aware_score_mean', ascending=False))


In [ ]:
def plot_metric(metric_mean, metric_std, title, ylabel, ascending=False):
    if summary.empty or metric_mean not in summary.columns:
        print(f'Missing metric: {metric_mean}')
        return
    data = summary.sort_values(metric_mean, ascending=ascending).copy()
    labels = data['module'].str.replace('_', '\n')
    yerr = data[metric_std] if metric_std in data.columns else None
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(labels, data[metric_mean], yerr=yerr, capsize=4)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_metric('labeled_test_mask_map50_mean', 'labeled_test_mask_map50_std', 'Labeled Test Mask mAP50', 'mAP50')
plot_metric('labeled_test_mask_map50_95_mean', 'labeled_test_mask_map50_95_std', 'Labeled Test Mask mAP50-95', 'mAP50-95')
plot_metric('healthy_test_healthy_mask_fp_rate_mean', 'healthy_test_healthy_mask_fp_rate_std', 'Healthy False Positive Rate', 'FP rate', ascending=True)
plot_metric('labeled_test_disease_box_miss_rate_mean', 'labeled_test_disease_box_miss_rate_std', 'Disease Miss Rate', 'miss rate', ascending=True)
plot_metric('healthy_aware_score_mean', 'healthy_aware_score_std', 'Healthy-Aware Score', 'score')


In [ ]:
if best_thresholds.empty:
    print('No best-threshold table yet. Run aggregate_multiseed_results.py after threshold sweep.')
else:
    cols = [
        'module', 'conf', 'iou', 'n_seeds', 'complete_3seed',
        'healthy_aware_score_mean', 'labeled_test_mask_map50_mean',
        'full_test_mask_map50_mean', 'healthy_test_healthy_mask_fp_rate_mean',
        'labeled_test_disease_box_miss_rate_mean', 'labeled_test_mask_count_mae_mean',
    ]
    display(best_thresholds[[c for c in cols if c in best_thresholds.columns]].sort_values('healthy_aware_score_mean', ascending=False))


## Final Notes

Use `custom_attention_multiseed/results/top4_final_ranking.md` as the source of truth for final conclusions. It is designed to avoid claiming a winner before all three seeds are available for all four modules.